In [1]:
# ============================================================
# CELDA 1
# Librerías
# ============================================================

import pandas as pd
import re
import unicodedata

from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 250)

In [2]:
# ============================================================
# CELDA 2
# Cargar matriz de estudios
# ============================================================

archivo_csv = "estudios_primarios.csv"

df = pd.read_csv(
    archivo_csv,
    sep=";",
    encoding="utf-8-sig"
)

print("Total de estudios incluidos:", len(df))
print("Número de columnas:", len(df.columns))

Total de estudios incluidos: 65
Número de columnas: 45


In [3]:
# ============================================================
# CELDA 3
# Seleccionar evidencia directamente asociada con RQ3
#
# Se consideran:
#   - Principal
#   - Sí
#
# Las categorías Parcial / No / No aborda se mantienen
# fuera de la síntesis principal.
# ============================================================

df_rq3 = df[
    df["RQ3"].isin([
        "Principal",
        "Sí"
    ])
].copy()

print("Distribución original de RQ3:")
print(df["RQ3"].value_counts())

print(
    "\nEstudios seleccionados para RQ3:",
    len(df_rq3)
)

Distribución original de RQ3:
RQ3
Principal                                             37
Parcial                                               15
No                                                     4
Sí                                                     3
No aborda                                              3
Parcial (operación)                                    1
Parcial (mecanismo de verificación no seguridad)       1
Parcial (verificación arquitectónica no seguridad)     1
Name: count, dtype: int64

Estudios seleccionados para RQ3: 40


In [4]:
# ============================================================
# CELDA 4
# Revisar campos directamente relacionados con RQ3
# ============================================================

columnas_rq3 = [
    "ID",
    "Título",
    "Propiedad / objetivo de seguridad",
    "Mecanismo de verificación",
    "Criterios / reglas verificadas",
    "Métricas",
    "RQ3"
]

display(
    df_rq3[columnas_rq3]
    .sort_values("ID")
)

,ID,Título,Propiedad / objetivo de seguridad,Mecanismo de verificación,Criterios / reglas verificadas,Métricas,RQ3
2,3,A model-driven knowledge graph treatment for process-centric cyber threat mitigation,"Secure-by-design, mitigación de amenazas, trazabilidad, auditoría, confidencialidad, integridad, disponibilidad, autenticación y no repudio","Consultas SPARQL de competencia, scoring de exposición a amenazas, comprobación de cobertura de mitigaciones, análisis de trust boundaries y evaluación GraphRAG con ground truth",Relaciones proceso--DFD--amenaza--mitigación; amenazas susceptibles; mitigaciones aplicadas; cruces de límites de confianza; cobertura y puntuación de mitigaciones; trazabilidad semántica,"Faithfulness, answer relevancy, context precision, context recall, F1 retrieval, F1 generation, porcentaje de respuestas correctas/parciales/incorrectas, tiempo y costo",Sí
6,9,An ontology-based secure design framework for graph-based databases,Control de acceso; autorización; protección/confidencialidad de información sensible; consistencia de políticas de seguridad,"Razonamiento OWL/DL para detectar inconsistencias, validación de reglas y consultas sobre Knowledge Graph; comprobación de las políticas generadas en las tecnologías destino",Restricciones de dominio/rango; clases disjuntas; jerarquías de roles; transitividad; consistencia entre tipo de regla y elemento/campo; privilegios Grant/Deny; condiciones de acceso,No presenta Precision/Recall u otras métricas de seguridad. FOOPS! evalúa FAIR: máxima puntuación en 17 de 24 pruebas,Sí
7,10,Automated regulatory compliance for AI systems in the security domain: The case of dual-use deployment,Privacidad; confidencialidad; control de acceso; cifrado; ciberseguridad; robustez; accountability; trazabilidad; logging; gestión de incidentes; seguridad por diseño,"Validación estructural mediante SPARQL, comprobación de tipado, dominio, asociaciones con fases del ciclo de vida y procedencia regulatoria; razonamiento semántico. SHACL aparece como mecanismo de automatización/validación que debe profundizarse",Correcta clasificación de requisitos; appliesToDomain; asociaciones con ciclo de vida; derivedFrom; categoría privacidad/seguridad; presencia de referencias regulatorias; consistencia y completitud estructural,"No presenta métricas estándar como Precision/Recall/F1. La evaluación se basa principalmente en completitud estructural, resultados de consultas, trazabilidad y capacidad de comparación entre dominios",Sí
9,13,CONSENT: A Software Architecture for Dynamic and Secure Consent Management,"Privacidad, trazabilidad, auditabilidad, integridad del consentimiento, revocabilidad y cumplimiento",Compliance scoring frente a artículos GDPR; verificación independiente; XAI; human-in-the-loop; blockchain hashing/revocation logging,Coherencia y cumplimiento del formulario; correspondencia con obligaciones regulatorias; necesidad de revisión humana en decisiones sensibles,"Coherencia, concisión, factualidad, fluidez, relevancia; tiempo/esfuerzo; compliance score; latencia",Principal
14,19,Leveraging LLMs and Knowledge Graphs to Design Secure Automation Systems,"Cumplimiento IEC 62443; autenticación/autorización, segmentación, confidencialidad/integridad y security level",Q&A grounded en KG; contraste con respuestas expertas; baseline SPARQL; cross-check del LLM con KG,Requisitos IEC 62443-3-3 aplicables al diseño; consistencia factual y semántica; correspondencia arquitectura-requisito,Accuracy 90% (closed/MC); factual consistency 79%; semantic similarity 84%; exact phrasing 32%; length similarity 69%; usabilidad,Principal
15,21,"LLM-Driven, Self-Improving Framework for Security Test Automation: Leveraging Karate DSL for Augmented API Resilience",Autorización correcta a nivel de objeto; resiliencia/seguridad API,Ejecución de tests de seguridad; syntax/compatibility verifier; resultados de Karate; refinamiento tras errores,Sintaxis Karate; autenticación; assertions; cobertura específica de 

In [5]:
# ============================================================
# CELDA 5
# Leer bibliografía LaTeX
# ============================================================

archivo_bib = "bibliografia.md"

texto_bib = Path(
    archivo_bib
).read_text(
    encoding="utf-8",
    errors="replace"
)

patron_bibitem = re.compile(
    r"\\bibitem\{([^}]+)\}\s*"
    r"(.*?)"
    r"(?=\\bibitem\{|\Z)",
    flags=re.DOTALL
)

referencias = []

for clave, bloque in patron_bibitem.findall(texto_bib):

    titulo_match = re.search(
        r"\\textit\{(.*?)\};",
        bloque,
        flags=re.DOTALL
    )

    if titulo_match:

        referencias.append({
            "BibKey": clave,
            "Título_bib": titulo_match.group(1).strip()
        })

bibliografia = pd.DataFrame(referencias)

print(
    "Referencias encontradas:",
    len(bibliografia)
)

display(bibliografia.head())

Referencias encontradas: 65


,BibKey,Título_bib
0,Spelten2024,A Methodology for Integrating Hierarchical VMAP-Data Structures into an Ontology Using Semantically Represented Analyses
1,Sun2020,A Model Driven Approach to Constructing Knowledge Graph from Relational Database
2,Chis2026,A model-driven knowledge graph treatment for process-centric cyber threat mitigation
3,Khadem2025,"Aligning DevOps and Microservice Architecture: Empirical Mapping, Taxonomy, and RAG-Based Decision Support"
4,Hadjichristofi2026,An Empirical Evaluation of Large Language Models Applying Software Architectural Patterns


In [6]:
# ============================================================
# CELDA 6
# Normalización de títulos para realizar el match
# ============================================================

def normalizar_titulo(texto):

    texto = str(texto)

    # Eliminar acentos
    texto = unicodedata.normalize(
        "NFKD",
        texto
    )

    texto = "".join(
        c for c in texto
        if not unicodedata.combining(c)
    )

    # Normalizar caracteres
    texto = (
        texto
        .lower()
        .replace("", "'")
        .replace("–", "-")
        .replace("—", "-")
    )

    # Mantener solo letras y números
    texto = re.sub(
        r"[^a-z0-9]+",
        " ",
        texto
    )

    texto = re.sub(
        r"\s+",
        " ",
        texto
    )

    return texto.strip()

In [7]:
# ============================================================
# CELDA 7
# Match: matriz <-> bibliografía
# ============================================================

bibliografia["titulo_normalizado"] = (
    bibliografia["Título_bib"]
    .apply(normalizar_titulo)
)

df_rq3["titulo_normalizado"] = (
    df_rq3["Título"]
    .apply(normalizar_titulo)
)

diccionario_bib = dict(
    zip(
        bibliografia["titulo_normalizado"],
        bibliografia["BibKey"]
    )
)

df_rq3["BibKey"] = (
    df_rq3["titulo_normalizado"]
    .map(diccionario_bib)
)

sin_match = df_rq3[
    df_rq3["BibKey"].isna()
]

print(
    "Estudios RQ3 sin referencia:",
    len(sin_match)
)

if len(sin_match) > 0:
    display(
        sin_match[
            ["ID", "Título"]
        ]
    )
else:
    print(
        "✓ Los 40 estudios tienen correspondencia bibliográfica."
    )

Estudios RQ3 sin referencia: 0
✓ Los 40 estudios tienen correspondencia bibliográfica.


### TABLA 1 — Mecanismos y propiedades verificadas
La síntesis produce seis familias principales.\
— Definir categorías de mecanismos

In [8]:
# ============================================================
# CELDA 8
# Categorías de mecanismos de verificación
#
# Cada estudio se asigna a su mecanismo dominante.
# ============================================================

categorias_mecanismos = {

    "Validación semántica y verificación basada en conocimiento": [
        3, 9, 10, 19, 75, 79
    ],

    "Pruebas de seguridad, cumplimiento y verificación humana": [
        13, 21
    ],

    "Evaluación de RAG/LLM, recuperación y grounding": [
        32, 38, 46, 49, 52,
        53, 55, 67, 69, 81
    ],

    "Detección de anomalías, fallos y localización de causa raíz": [
        41, 45, 50, 51, 57,
        61, 62, 73, 74, 83,
        84, 86, 87
    ],

    "Pruebas de rendimiento y cumplimiento de SLA": [
        47, 58, 77
    ],

    "Verificación de calidad, reglas, cobertura y smells": [
        39, 44, 56, 59, 60, 66
    ]
}

In [9]:
# ============================================================
# CELDA 9
# Síntesis de propiedades u objetivos asociados
# ============================================================

propiedades_mecanismos = {

    "Validación semántica y verificación basada en conocimiento":
        "Secure-by-Design; confidencialidad; integridad; "
        "disponibilidad; autenticación/autorización; "
        "cumplimiento; trazabilidad y consistencia",

    "Pruebas de seguridad, cumplimiento y verificación humana":
        "Privacidad; integridad y auditabilidad del consentimiento; "
        "autorización; seguridad y resiliencia de APIs",

    "Evaluación de RAG/LLM, recuperación y grounding":
        "Confiabilidad; factualidad; auditabilidad; trazabilidad; "
        "privacidad/confidencialidad en escenarios seguros "
        "y acceso controlado a información",

    "Detección de anomalías, fallos y localización de causa raíz":
        "Disponibilidad; confiabilidad; resiliencia; robustez "
        "y estabilidad operacional",

    "Pruebas de rendimiento y cumplimiento de SLA":
        "Disponibilidad; confiabilidad; cumplimiento de SLA; "
        "rendimiento; latencia y eficiencia",

    "Verificación de calidad, reglas, cobertura y smells":
        "Consistencia y correctitud; robustez funcional; "
        "mantenibilidad; modularidad y calidad arquitectónica"
}

In [10]:
# ============================================================
# CELDA 10
# Control de consistencia
# ============================================================

ids_mecanismos = [
    estudio
    for ids in categorias_mecanismos.values()
    for estudio in ids
]

ids_rq3 = set(df_rq3["ID"])
ids_categoria = set(ids_mecanismos)

print("Estudios RQ3:", len(ids_rq3))
print("Estudios categorizados:", len(ids_categoria))

print(
    "Faltantes:",
    sorted(ids_rq3 - ids_categoria)
)

print(
    "Adicionales:",
    sorted(ids_categoria - ids_rq3)
)

duplicados = [
    x
    for x in ids_mecanismos
    if ids_mecanismos.count(x) > 1
]

print(
    "Duplicados:",
    sorted(set(duplicados))
)

Estudios RQ3: 40
Estudios categorizados: 40
Faltantes: []
Adicionales: []
Duplicados: []


In [11]:
# ============================================================
# CELDA 11
# Tabla:
# mecanismos + propiedades + referencias + total
# ============================================================

filas = []

for mecanismo, ids in categorias_mecanismos.items():

    estudios = (
        df_rq3[
            df_rq3["ID"].isin(ids)
        ]
        .sort_values("ID")
    )

    referencias = (
        "\\cite{"
        + ",".join(estudios["BibKey"].tolist())
        + "}"
    )

    filas.append({

        "Mecanismo de verificación": mecanismo,

        "Propiedades / objetivos de seguridad":
            propiedades_mecanismos[mecanismo],

        "Estudios designados": referencias,

        "Total": len(estudios)
    })


tabla_rq3_mecanismos = pd.DataFrame(filas)

display(tabla_rq3_mecanismos)

,Mecanismo de verificación,Propiedades / objetivos de seguridad,Estudios designados,Total
0,Validación semántica y verificación basada en conocimiento,Secure-by-Design; confidencialidad; integridad; disponibilidad; autenticación/autorización; cumplimiento; trazabilidad y consistencia,"\cite{Chis2026,Paneque2024,Sabaliauskaite2026,Hosseini2025,VergaraVargas2024,Zhang2025b}",6
1,"Pruebas de seguridad, cumplimiento y verificación humana",Privacidad; integridad y auditabilidad del consentimiento; autorización; seguridad y resiliencia de APIs,"\cite{Zoi2026,MarianPasca2025}",2
2,"Evaluación de RAG/LLM, recuperación y grounding",Confiabilidad; factualidad; auditabilidad; trazabilidad; privacidad/confidencialidad en escenarios seguros y acceso controlado a información,"\cite{Ersoy2025,Awadid2025,Gijre2024,Guan2024,Su2025,Omrani2024,Mukherjee2025,Jin2025,Qiao2024,Thanh2025}",10
3,"Detección de anomalías, fallos y localización de causa raíz",Disponibilidad; confiabilidad; resiliencia; robustez y estabilidad operacional,"\cite{Hu2024,Tao2024,Chen2024,Zhang2025,Yao2023,Chen2024b,Wang2025,Yang2022,Zhang2023,Ding2024,Sun2025,Xu2024,Wang2020}",13
4,Pruebas de rendimiento y cumplimiento de SLA,Disponibilidad; confiabilidad; cumplimiento de SLA; rendimiento; latencia y eficiencia,"\cite{William2025,Mascia2025,Yamsani2026}",3
5,"Verificación de calidad, reglas, cobertura y smells",Consistencia y correctitud; robustez funcional; mantenibilidad; modularidad y calidad arquitectónica,"\cite{Mohammed2022,Oliveira2022,Rincon2025,Bacchiega2022,Gstur2024,Bacchiega2024}",6


### TABLA 2 — Criterios y métricas
— Categorizar criterios y métricas

In [12]:
# ============================================================
# CELDA 12
# Categorías principales de criterios/métricas
# ============================================================

categorias_metricas = {

    "Métricas de clasificación, detección y localización": [
        38, 41, 44, 45, 49,
        50, 51, 57, 58, 60,
        61, 62, 67, 73, 74,
        83, 84, 86, 87
    ],

    "Calidad de recuperación y generación (RAG/LLM)": [
        3, 19, 32, 46,
        52, 53, 55, 69
    ],

    "Cumplimiento semántico, normativo y de reglas": [
        9, 10, 13, 21,
        59, 66, 75, 79
    ],

    "Rendimiento, SLA y eficiencia de recursos": [
        47, 77
    ],

    "Cobertura, corrección funcional y ejecución": [
        39, 56, 81
    ]
}

In [13]:
# ============================================================
# CELDA 13
# Métricas representativas de cada categoría
# ============================================================

descripcion_metricas = {

    "Métricas de clasificación, detección y localización":
        "Accuracy, Precision, Recall, F1/F-measure, AUC/ROC-AUC, "
        "Top-k, P@k, A@k, MRR, Hits@n y precisión de localización",

    "Calidad de recuperación y generación (RAG/LLM)":
        "Faithfulness, answer relevancy, context precision, "
        "context recall, semantic similarity, correctness, "
        "citation coverage y métricas RAGAS",

    "Cumplimiento semántico, normativo y de reglas":
        "Consistencia ontológica, completitud estructural, "
        "cumplimiento de reglas/políticas, compliance score, "
        "cobertura de mitigaciones y validación cualitativa",

    "Rendimiento, SLA y eficiencia de recursos":
        "Latencia media/P90/P95, completion rate, success rate, "
        "timeouts, error rate, SLA violation rate y consumo energético",

    "Cobertura, corrección funcional y ejecución":
        "Cobertura de ciclos, código y endpoints; validez de tests; "
        "execution accuracy y component matching"
}

In [14]:
# ============================================================
# CELDA 14
# Verificar que cada estudio tenga una categoría
# ============================================================

ids_metricas = [
    estudio
    for ids in categorias_metricas.values()
    for estudio in ids
]

print(
    "Estudios categorizados:",
    len(set(ids_metricas))
)

print(
    "Faltantes:",
    sorted(
        set(df_rq3["ID"])
        - set(ids_metricas)
    )
)

print(
    "Duplicados:",
    sorted(
        set([
            x for x in ids_metricas
            if ids_metricas.count(x) > 1
        ])
    )
)

Estudios categorizados: 40
Faltantes: []
Duplicados: []


In [15]:
# ============================================================
# CELDA 15
# Tabla de criterios y métricas
# ============================================================

filas = []

for categoria, ids in categorias_metricas.items():

    estudios = (
        df_rq3[
            df_rq3["ID"].isin(ids)
        ]
        .sort_values("ID")
    )

    referencias = (
        "\\cite{"
        + ",".join(estudios["BibKey"].tolist())
        + "}"
    )

    filas.append({

        "Criterio / grupo de métricas": categoria,

        "Métricas empleadas":
            descripcion_metricas[categoria],

        "Estudios designados": referencias,

        "Total": len(estudios)
    })


tabla_rq3_metricas = pd.DataFrame(filas)

display(tabla_rq3_metricas)

,Criterio / grupo de métricas,Métricas empleadas,Estudios designados,Total
0,"Métricas de clasificación, detección y localización","Accuracy, Precision, Recall, F1/F-measure, AUC/ROC-AUC, Top-k, P@k, A@k, MRR, Hits@n y precisión de localización","\cite{Awadid2025,Hu2024,Oliveira2022,Tao2024,Guan2024,Chen2024,Zhang2025,Yao2023,Mascia2025,Gstur2024,Chen2024b,Wang2025,Jin2025,Yang2022,Zhang2023,Ding2024,Sun2025,Xu2024,Wang2020}",19
1,Calidad de recuperación y generación (RAG/LLM),"Faithfulness, answer relevancy, context precision, context recall, semantic similarity, correctness, citation coverage y métricas RAGAS","\cite{Chis2026,Hosseini2025,Ersoy2025,Gijre2024,Su2025,Omrani2024,Mukherjee2025,Qiao2024}",8
2,"Cumplimiento semántico, normativo y de reglas","Consistencia ontológica, completitud estructural, cumplimiento de reglas/políticas, compliance score, cobertura de mitigaciones y validación cualitativa","\cite{Paneque2024,Sabaliauskaite2026,Zoi2026,MarianPasca2025,Bacchiega2022,Bacchiega2024,VergaraVargas2024,Zhang2025b}",8
3,"Rendimiento, SLA y eficiencia de recursos","Latencia media/P90/P95, completion rate, success rate, timeouts, error rate, SLA violation rate y consumo energético","\cite{William2025,Yamsani2026}",2
4,"Cobertura, corrección funcional y ejecución","Cobertura de ciclos, código y endpoints; validez de tests; execution accuracy y component matching","\cite{Mohammed2022,Rincon2025,Thanh2025}",3


**Mostrar las referencias exactas**\
— Tabla 1 con referencias

In [16]:
# ============================================================
# CELDA 16
# Mostrar referencias de mecanismos
# ============================================================

for _, fila in tabla_rq3_mecanismos.iterrows():

    print("=" * 90)

    print(
        fila["Mecanismo de verificación"]
    )

    print(
        "Propiedades:",
        fila["Propiedades / objetivos de seguridad"]
    )

    print(
        "Estudios:",
        fila["Estudios designados"]
    )

    print(
        "Total:",
        fila["Total"]
    )

Validación semántica y verificación basada en conocimiento
Propiedades: Secure-by-Design; confidencialidad; integridad; disponibilidad; autenticación/autorización; cumplimiento; trazabilidad y consistencia
Estudios: \cite{Chis2026,Paneque2024,Sabaliauskaite2026,Hosseini2025,VergaraVargas2024,Zhang2025b}
Total: 6
Pruebas de seguridad, cumplimiento y verificación humana
Propiedades: Privacidad; integridad y auditabilidad del consentimiento; autorización; seguridad y resiliencia de APIs
Estudios: \cite{Zoi2026,MarianPasca2025}
Total: 2
Evaluación de RAG/LLM, recuperación y grounding
Propiedades: Confiabilidad; factualidad; auditabilidad; trazabilidad; privacidad/confidencialidad en escenarios seguros y acceso controlado a información
Estudios: \cite{Ersoy2025,Awadid2025,Gijre2024,Guan2024,Su2025,Omrani2024,Mukherjee2025,Jin2025,Qiao2024,Thanh2025}
Total: 10
Detección de anomalías, fallos y localización de causa raíz
Propiedades: Disponibilidad; confiabilidad; resiliencia; robustez y estab

**Exportación a LaTeX**\
— Función auxiliar

In [17]:
# ============================================================
# CELDA 17
# Escapar caracteres especiales de LaTeX
# ============================================================

def escapar_latex(texto):

    texto = str(texto)

    reemplazos = {
        "&": r"\&",
        "%": r"\%",
        "#": r"\#",
        "_": r"\_"
    }

    for original, nuevo in reemplazos.items():
        texto = texto.replace(
            original,
            nuevo
        )

    return texto

**Exportar Tabla 1**\
— Generar LaTeX de mecanismos y propiedades

In [18]:
# ============================================================
# CELDA 18
# Tabla LaTeX:
# mecanismos + propiedades + estudios + total
# ============================================================

def latex_mecanismos_rq3(tabla):

    lineas = []

    lineas.append(r"\begin{table}[H]")

    lineas.append(
        r"\caption{Mecanismos de verificación y propiedades "
        r"evaluadas en los estudios que responden a RQ3."
        r"\label{tab:rq3-mecanismos}}"
    )

    lineas.append(
        r"%\isPreprints{\centering}{%"
    )

    lineas.append(
        r"\begin{adjustwidth}{-\extralength}{0cm}"
    )

    lineas.append(r"%}")

    lineas.append(
        r"%\isPreprints{\begin{tabularx}{\textwidth}{CCCC}}{%"
    )

    lineas.append(
        r"\begin{tabularx}{\fulllength}"
        r"{p{0.23\fulllength} "
        r"p{0.29\fulllength} "
        r"X "
        r"p{0.06\fulllength}}"
    )

    lineas.append(r"%}")

    lineas.append(r"\toprule")

    lineas.append(
        r"\textbf{Mecanismo de verificación} & "
        r"\textbf{Propiedades / objetivos} & "
        r"\textbf{Estudios designados} & "
        r"\textbf{Total} \\"
    )

    lineas.append(r"\midrule")
    lineas.append("")

    for _, fila in tabla.iterrows():

        mecanismo = escapar_latex(
            fila["Mecanismo de verificación"]
        )

        propiedades = escapar_latex(
            fila["Propiedades / objetivos de seguridad"]
        )

        citas = fila["Estudios designados"]

        total = fila["Total"]

        lineas.append(
            f"{mecanismo} & "
            f"{propiedades} & "
            f"{citas} & "
            f"{total} \\\\"
        )

        lineas.append("")

    lineas.append(r"\midrule")
    lineas.append(
        r"\textbf{Total} & & & \textbf{40} \\"
    )
    lineas.append(r"\bottomrule")

    lineas.append(r"\end{tabularx}")

    lineas.append(
        r"%\isPreprints{}{%"
    )

    lineas.append(
        r"\end{adjustwidth}"
    )

    lineas.append(r"%}")
    lineas.append(r"\end{table}")

    return "\n".join(lineas)


latex_tabla_mecanismos = (
    latex_mecanismos_rq3(
        tabla_rq3_mecanismos
    )
)

print(latex_tabla_mecanismos)

\begin{table}[H]
\caption{Mecanismos de verificación y propiedades evaluadas en los estudios que responden a RQ3.\label{tab:rq3-mecanismos}}
%\isPreprints{\centering}{%
\begin{adjustwidth}{-\extralength}{0cm}
%}
%\isPreprints{\begin{tabularx}{\textwidth}{CCCC}}{%
\begin{tabularx}{\fulllength}{p{0.23\fulllength} p{0.29\fulllength} X p{0.06\fulllength}}
%}
\toprule
\textbf{Mecanismo de verificación} & \textbf{Propiedades / objetivos} & \textbf{Estudios designados} & \textbf{Total} \\
\midrule

Validación semántica y verificación basada en conocimiento & Secure-by-Design; confidencialidad; integridad; disponibilidad; autenticación/autorización; cumplimiento; trazabilidad y consistencia & \cite{Chis2026,Paneque2024,Sabaliauskaite2026,Hosseini2025,VergaraVargas2024,Zhang2025b} & 6 \\

Pruebas de seguridad, cumplimiento y verificación humana & Privacidad; integridad y auditabilidad del consentimiento; autorización; seguridad y resiliencia de APIs & \cite{Zoi2026,MarianPasca2025} & 2 \\

Eval

**Exportar Tabla 2**\
— Generar LaTeX de criterios y métricas

In [19]:
# ============================================================
# CELDA 19
# Tabla LaTeX:
# criterios/métricas + referencias + total
# ============================================================

def latex_metricas_rq3(tabla):

    lineas = []

    lineas.append(r"\begin{table}[H]")

    lineas.append(
        r"\caption{Criterios y métricas empleados para evaluar "
        r"el cumplimiento en los estudios que responden a RQ3."
        r"\label{tab:rq3-metricas}}"
    )

    lineas.append(
        r"%\isPreprints{\centering}{%"
    )

    lineas.append(
        r"\begin{adjustwidth}{-\extralength}{0cm}"
    )

    lineas.append(r"%}")

    lineas.append(
        r"%\isPreprints{\begin{tabularx}{\textwidth}{CCCC}}{%"
    )

    lineas.append(
        r"\begin{tabularx}{\fulllength}"
        r"{p{0.23\fulllength} "
        r"p{0.31\fulllength} "
        r"X "
        r"p{0.06\fulllength}}"
    )

    lineas.append(r"%}")
    lineas.append(r"\toprule")

    lineas.append(
        r"\textbf{Criterio / grupo de métricas} & "
        r"\textbf{Métricas empleadas} & "
        r"\textbf{Estudios designados} & "
        r"\textbf{Total} \\"
    )

    lineas.append(r"\midrule")
    lineas.append("")

    for _, fila in tabla.iterrows():

        categoria = escapar_latex(
            fila["Criterio / grupo de métricas"]
        )

        metricas = escapar_latex(
            fila["Métricas empleadas"]
        )

        citas = fila["Estudios designados"]

        total = fila["Total"]

        lineas.append(
            f"{categoria} & "
            f"{metricas} & "
            f"{citas} & "
            f"{total} \\\\"
        )

        lineas.append("")

    lineas.append(r"\midrule")
    lineas.append(
        r"\textbf{Total} & & & \textbf{40} \\"
    )

    lineas.append(r"\bottomrule")
    lineas.append(r"\end{tabularx}")

    lineas.append(
        r"%\isPreprints{}{%"
    )

    lineas.append(
        r"\end{adjustwidth}"
    )

    lineas.append(r"%}")
    lineas.append(r"\end{table}")

    return "\n".join(lineas)


latex_tabla_metricas = (
    latex_metricas_rq3(
        tabla_rq3_metricas
    )
)

print(latex_tabla_metricas)

\begin{table}[H]
\caption{Criterios y métricas empleados para evaluar el cumplimiento en los estudios que responden a RQ3.\label{tab:rq3-metricas}}
%\isPreprints{\centering}{%
\begin{adjustwidth}{-\extralength}{0cm}
%}
%\isPreprints{\begin{tabularx}{\textwidth}{CCCC}}{%
\begin{tabularx}{\fulllength}{p{0.23\fulllength} p{0.31\fulllength} X p{0.06\fulllength}}
%}
\toprule
\textbf{Criterio / grupo de métricas} & \textbf{Métricas empleadas} & \textbf{Estudios designados} & \textbf{Total} \\
\midrule

Métricas de clasificación, detección y localización & Accuracy, Precision, Recall, F1/F-measure, AUC/ROC-AUC, Top-k, P@k, A@k, MRR, Hits@n y precisión de localización & \cite{Awadid2025,Hu2024,Oliveira2022,Tao2024,Guan2024,Chen2024,Zhang2025,Yao2023,Mascia2025,Gstur2024,Chen2024b,Wang2025,Jin2025,Yang2022,Zhang2023,Ding2024,Sun2025,Xu2024,Wang2020} & 19 \\

Calidad de recuperación y generación (RAG/LLM) & Faithfulness, answer relevancy, context precision, context recall, semantic similarity, c

**— Guardar ambas tablas .tex**

In [20]:
# ============================================================
# CELDA 20
# Exportar las dos tablas
# ============================================================

with open(
    "tabla_rq3_mecanismos.tex",
    "w",
    encoding="utf-8"
) as archivo:

    archivo.write(
        latex_tabla_mecanismos
    )


with open(
    "tabla_rq3_metricas.tex",
    "w",
    encoding="utf-8"
) as archivo:

    archivo.write(
        latex_tabla_metricas
    )


print("Archivos generados:")
print(" - tabla_rq3_mecanismos.tex")
print(" - tabla_rq3_metricas.tex")

Archivos generados:
 - tabla_rq3_mecanismos.tex
 - tabla_rq3_metricas.tex


— Exportar trazabilidad a Excel

In [21]:
# ============================================================
# CELDA 21
# Matriz de trazabilidad de RQ3
# ============================================================

mecanismo_por_id = {}

for categoria, ids in categorias_mecanismos.items():

    for id_estudio in ids:

        mecanismo_por_id[id_estudio] = categoria


metrica_por_id = {}

for categoria, ids in categorias_metricas.items():

    for id_estudio in ids:

        metrica_por_id[id_estudio] = categoria


df_rq3["Categoría mecanismo RQ3"] = (
    df_rq3["ID"].map(mecanismo_por_id)
)

df_rq3["Categoría métricas RQ3"] = (
    df_rq3["ID"].map(metrica_por_id)
)


trazabilidad_rq3 = df_rq3[
    [
        "ID",
        "BibKey",
        "Título",
        "Propiedad / objetivo de seguridad",
        "Mecanismo de verificación",
        "Categoría mecanismo RQ3",
        "Criterios / reglas verificadas",
        "Métricas",
        "Categoría métricas RQ3"
    ]
].sort_values("ID")


trazabilidad_rq3.to_excel(
    "trazabilidad_RQ3.xlsx",
    index=False
)

display(trazabilidad_rq3)

print(
    "Archivo generado: trazabilidad_RQ3.xlsx"
)

,ID,BibKey,Título,Propiedad / objetivo de seguridad,Mecanismo de verificación,Categoría mecanismo RQ3,Criterios / reglas verificadas,Métricas,Categoría métricas RQ3
2,3,Chis2026,A model-driven knowledge graph treatment for process-centric cyber threat mitigation,"Secure-by-design, mitigación de amenazas, trazabilidad, auditoría, confidencialidad, integridad, disponibilidad, autenticación y no repudio","Consultas SPARQL de competencia, scoring de exposición a amenazas, comprobación de cobertura de mitigaciones, análisis de trust boundaries y evaluación GraphRAG con ground truth",Validación semántica y verificación basada en conocimiento,Relaciones proceso--DFD--amenaza--mitigación; amenazas susceptibles; mitigaciones aplicadas; cruces de límites de confianza; cobertura y puntuación de mitigaciones; trazabilidad semántica,"Faithfulness, answer relevancy, context precision, context recall, F1 retrieval, F1 generation, porcentaje de respuestas correctas/parciales/incorrectas, tiempo y costo",Calidad de recuperación y generación (RAG/LLM)
6,9,Paneque2024,An ontology-based secure design framework for graph-based databases,Control de acceso; autorización; protección/confidencialidad de información sensible; consistencia de políticas de seguridad,"Razonamiento OWL/DL para detectar inconsistencias, validación de reglas y consultas sobre Knowledge Graph; comprobación de las políticas generadas en las tecnologías destino",Validación semántica y verificación basada en conocimiento,Restricciones de dominio/rango; clases disjuntas; jerarquías de roles; transitividad; consistencia entre tipo de regla y elemento/campo; privilegios Grant/Deny; condiciones de acceso,No presenta Precision/Recall u otras métricas de seguridad. FOOPS! evalúa FAIR: máxima puntuación en 17 de 24 pruebas,"Cumplimiento semántico, normativo y de reglas"
7,10,Sabaliauskaite2026,Automated regulatory compliance for AI systems in the security domain: The case of dual-use deployment,Privacidad; confidencialidad; control de acceso; cifrado; ciberseguridad; robustez; accountability; trazabilidad; logging; gestión de incidentes; seguridad por diseño,"Validación estructural mediante SPARQL, comprobación de tipado, dominio, asociaciones con fases del ciclo de vida y procedencia regulatoria; razonamiento semántico. SHACL aparece como mecanismo de automatización/validación que debe profundizarse",Validación semántica y verificación basada en conocimiento,Correcta clasificación de requisitos; appliesToDomain; asociaciones con ciclo de vida; derivedFrom; categoría privacidad/seguridad; presencia de referencias regulatorias; consistencia y completitud estructural,"No presenta métricas estándar como Precision/Recall/F1. La evaluación se basa principalmente en completitud estructural, resultados de consultas, trazabilidad y capacidad de comparación entre dominios","Cumplimiento semántico, normativo y de reglas"
9,13,Zoi2026,CONSENT: A Software Architecture for Dynamic and Secure Consent Management,"Privacidad, trazabilidad, auditabilidad, integridad del consentimiento, revocabilidad y cumplimiento",Compliance scoring frente a artículos GDPR; verificación independiente; XAI; human-in-the-loop; blockchain hashing/revocation logging,"Pruebas de seguridad, cumplimiento y verificación humana",Coherencia y cumplimiento del formulario; correspondencia con obligaciones regulatorias; necesidad de revisión humana en decisiones sensibles,"Coherencia, concisión, factualidad, fluidez, relevancia; tiempo/esfuerzo; compliance score; latencia","Cumplimiento semántico, normativo y de reglas"
14,19,Hosseini2025,Leveraging LLMs and Knowledge Graphs to Design Secure Automation Systems,"Cumplimiento IEC 62443; autenticación/autorización, segmentación, confidencialidad/integridad y security level",Q&A grounded en KG; contraste con respuestas expertas; baseline SPARQL; cross-check del LLM con KG,Validación semántica y verificación basada en conocimiento,Requisitos IEC 62443-3-3 aplicables al diseño; consiste

Archivo generado: trazabilidad_RQ3.xlsx
